# Long-reasoning pilot on Qwen3-0.6B (Colab)

The same run as `chains/long_4b.sh` (the pilot registered in `PREREGISTRATION.md`), with `BASE_MODEL=Qwen/Qwen3-0.6B`:

1. **calibration**: the base model on the 962 calibration questions (2 rollouts each, cap 32,768 tokens), then answers forced at up to 12 points of every rollout -> the "settled point" of each rollout;
2. **pilot**: the base model on the 300 validation questions (4 rollouts each), training rows, the two ways of training ("trace", "decision"), both adapters on the validation questions, and the fixed rule (overall >= -1.0 pp, every length bucket >= -3.0 pp).

**Before you start**
- Runtime -> Change runtime type -> a GPU with bf16: **A100, H100, L4 or RTX PRO 6000 (G4)**. A T4 does not work (no bf16).
- In Google Drive, the folder `qwen06b_long` holds `long_code_06b.tar.gz`, `long_cal_pools.tar.gz` and `long_pools.tar.gz`.

**Run** all cells in order (Runtime -> Run all). The chain runs in the background; cell 6 shows how far it is (run it again any time).
Everything it writes goes to `qwen06b_long/work` in Drive. If the runtime stops, connect again and **run all cells again**: finished parts are skipped and the chain continues.

**Results**: at the end the archive `qwen06b_long/results_06b_long.tar.gz` appears in Drive (also at a stop with an error); the last cell shows the result.
Keep the browser tab open: without Colab Pro+ background execution the runtime stops when the tab is closed for a while.

In [ ]:
# 1. Settings and Google Drive
DRIVE_DIR = "/content/drive/MyDrive/qwen06b_long"   # the folder with the three archives
TAG = "06b"
MODEL = "Qwen/Qwen3-0.6B"

import os, subprocess, pathlib
from google.colab import drive
drive.mount("/content/drive")
D = pathlib.Path(DRIVE_DIR)
missing = [f for f in ("long_code_06b.tar.gz", "long_cal_pools.tar.gz", "long_pools.tar.gz") if not (D / f).exists()]
assert not missing, f"missing in {D}: {missing}"
(D / "work" / "runs").mkdir(parents=True, exist_ok=True)
(D / "work" / "checkpoints").mkdir(parents=True, exist_ok=True)
print("Drive folder OK:", sorted(p.name for p in D.iterdir()))

In [ ]:
# 2. GPU check: needs bf16 (compute capability 8.0 or more)
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(out)
assert out, "no GPU: Runtime -> Change runtime type -> GPU"
cap = float(out.split(",")[2])
assert cap >= 8.0, "this GPU has no bf16 (a T4?). Choose A100, H100, L4 or RTX PRO 6000."

In [ ]:
%%bash -s "$DRIVE_DIR"
# 3. Code, data and packages (about 5-10 minutes; again after every new runtime)
set -e
D=$1; R=/content/repo
if [ ! -d $R/stopping ]; then
    mkdir -p $R && cd $R
    tar xzf $D/long_code_06b.tar.gz
    tar xzf $D/long_cal_pools.tar.gz
    tar xzf $D/long_pools.tar.gz
    ln -sfn $D/work/runs runs
    ln -sfn $D/work/checkpoints checkpoints
fi
cd $R && ls
(apt-get -qq update && apt-get -qq install -y bubblewrap util-linux) > /dev/null 2>&1 || echo "apt: bubblewrap not installed (the fallback sandbox is used)"
pip install -q "vllm==0.34.0" "transformers==5.22.2" "peft==0.21.3" "math-verify[antlr4_13_2]" emoji syllapy pandas pyarrow \
    absl-py langdetect nltk immutabledict 2>&1 | grep -v -i "warn\|dependency resolver\|incompatible" | tail -5 || true
python -c "import torch, vllm, transformers, peft; assert torch.cuda.is_available(), 'torch does not see the GPU (driver too old for this torch?)'; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| bf16', torch.cuda.is_bf16_supported())"

In [ ]:
# 4. Sandbox for the model-written programs (code questions are judged by running them)
#    bubblewrap if it works here; otherwise the fallback (user "nobody", see stopping/code_tasks.py),
#    used only after checking that "nobody" can write neither the Drive folder nor the code.
os.chdir("/content/repo")
def self_test(mode):
    env = dict(os.environ, SANDBOX=mode)
    r = subprocess.run(["python", "-c", "import sys; sys.path.insert(0, '.'); from stopping.code_tasks import self_test; print(self_test() or 'OK')"],
                       capture_output=True, text=True, env=env)
    return (r.stdout.strip().splitlines() or [r.stderr.strip()[-300:]])[-1]
SANDBOX = "bwrap"
result = self_test("bwrap")
print("bubblewrap:", result)
if result != "OK":
    probe = subprocess.run(["setpriv", "--reuid=65534", "--regid=65534", "--clear-groups", "--", "sh", "-c",
                            f"touch '{DRIVE_DIR}/work/.nobody' 2>/dev/null && echo DRIVE; touch /content/repo/.nobody 2>/dev/null && echo REPO; true"],
                           capture_output=True, text=True)
    writable = probe.stdout.split()
    assert not writable, f"user nobody can write {writable}: the fallback is not safe here - stop and report this"
    SANDBOX = "user"
    result = self_test("user")
    print("fallback (user nobody):", result)
assert result == "OK", "no working sandbox: code questions cannot be judged - stop and report this"
print("sandbox:", SANDBOX)

In [ ]:
# 5. Start the chain in the background (does nothing if it is already running)
os.chdir("/content/repo")
running = subprocess.run(["pgrep", "-f", "chains/long_4b.sh"], capture_output=True, text=True).stdout.split()
if running:
    print("already running, pids", running)
else:
    log = f"{DRIVE_DIR}/long_{TAG}.log"
    if os.path.exists(log):      # the chain packs the log from the repo folder; keep the history of earlier runtimes
        subprocess.run(["cp", log, f"long_{TAG}.log"])
    env = (f"VENV=none NO_LOCK=1 SANDBOX={SANDBOX} TAG={TAG} BASE_MODEL={MODEL} JUDGE_WORKERS=2 "
           f"HF_HOME=/content/hf_cache GPU_FREE_MIB=4000")
    command = (f"{env} bash chains/long_4b.sh 2>&1 | tee -a long_{TAG}.log >> '{log}'; "
               f"cp results_{TAG}_long.tar.gz '{DRIVE_DIR}/' 2>/dev/null; echo CHAIN-EXITED >> '{log}'")
    subprocess.Popen(["bash", "-c", command], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL)
    print("started; the log is", log)

In [ ]:
# 6. Progress (run again whenever you like). The first part (calibration rollouts, 1-3 h) prints nothing
#    until it ends: the GPU line shows that it works.
log = f"{DRIVE_DIR}/long_{TAG}.log"
print(subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
alive = subprocess.run(["pgrep", "-f", "chains/long_4b.sh"], capture_output=True, text=True).stdout.split()
print("chain running" if alive else "chain NOT running")
print(subprocess.run(["tail", "-25", log], capture_output=True, text=True).stdout)

In [ ]:
# 7. Result (after "CHAIN-EXITED" in the log). The archive is in Drive: qwen06b_long/results_06b_long.tar.gz
import glob
for f in (f"/content/repo/runs/{TAG}_long/pilot.txt", f"/content/repo/runs/{TAG}_long_cal/readouts_report.txt"):
    if os.path.exists(f):
        print("==", f); print(open(f).read())
print(glob.glob(f"{DRIVE_DIR}/results_*.tar.gz"))
# from google.colab import files; files.download(f"{DRIVE_DIR}/results_{TAG}_long.tar.gz")   # also straight to this computer